# 2. DATA CLEANING

**Objectif**: Corriger, transformer et encoder les données pour l'EDA et le ML.

**Règle d'or**: On ne modifie jamais le df_draw. Toutes les transformations sont appliquées sur une copie et tracées dans ce notebook.

**Fonctions utilisées**: src/cleaning.py

In [30]:
# Import

import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from src.cleaning import (
    fix_TotalCharges,
    encode_target,
    encode_bin_columns,
    encode_nominal_cols,
    run_cleaning_pipeline,
    validate_clean_df
)

pd.set_option('display.max.rows', 10)
DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [31]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Données brutes chargée: \n{df_raw.shape}')

Données brutes chargée: 
(7043, 21)


## 1. Correction de TotalChrges

In [32]:
df = fix_TotalCharges(df_raw)

avant = df_raw['TotalCharges'].dtype
apres = df['TotalCharges'].dtype
nan_apres = df['TotalCharges'].isnull().sum()

print(f'Type avant: {avant} \nType après: {apres}')
print(f'Valeur manquantes après: {nan_apres}')
print(f'Médiane utilisée pour imputation: {df["TotalCharges"].median():.2f}€')

Type avant: str 
Type après: float64
Valeur manquantes après: 0
Médiane utilisée pour imputation: 1397.47€


## 2. Encodage de la variable cible

In [33]:
df = encode_target(df)

print(f'Distribution après encodage')
print(df['Churn'].value_counts())
print('0 = non-churn et 1 = churn')

Distribution après encodage
Churn
0    5174
1    1869
Name: count, dtype: int64
0 = non-churn et 1 = churn


## 3. Suppression de customerID

In [34]:
df = df.drop('customerID', axis=1)
print(f'Shape après la suppression de customerID: {df.shape}')

Shape après la suppression de customerID: (7043, 20)


## 4. Encodage des variables binaires

In [35]:
df = encode_bin_columns(df)

## Vérification : toutes les colonnes doivent être en int avec valeurs 0 ou 1
cols_to_check = ['gender', 'Partner', 'Dependents', 'PhoneService',
                 'MultipleLines', 'OnlineSecurity', 'TechSupport']
print("Vérification encodage binaire: ")
for col in cols_to_check:
    val = df[col].unique().tolist()
    if set(val).issubset({0, 1}):
        statut = 'Ok'
    else:
        statut = 'Not ok'

    print(f'{statut} {col:20s} valeurs: {sorted(val)}')


Vérification encodage binaire: 
Ok gender               valeurs: [0, 1]
Ok Partner              valeurs: [0, 1]
Ok Dependents           valeurs: [0, 1]
Ok PhoneService         valeurs: [0, 1]
Ok MultipleLines        valeurs: [0, 1]
Ok OnlineSecurity       valeurs: [0, 1]
Ok TechSupport          valeurs: [0, 1]


## 5. One-Hot Encoding (variables nominales)

In [36]:
df = encode_nominal_cols(df)

new_cols = [col for col in df.columns if any(i in col for i in ["InternetService", "Contract", "PaymentMethod"])]
print(f'Shape après OHE: {df.shape}')
print(f'\nNouvelles colonnes créées ({len(new_cols)})')
for col in new_cols:
    print(f'{col}')

Shape après OHE: (7043, 27)

Nouvelles colonnes créées (10)
InternetService_DSL
InternetService_Fiber optic
InternetService_No
Contract_Month-to-month
Contract_One year
Contract_Two year
PaymentMethod_Bank transfer (automatic)
PaymentMethod_Credit card (automatic)
PaymentMethod_Electronic check
PaymentMethod_Mailed check


## 6. Validation finale

In [38]:
validate_clean_df(df)
df.head(5)

Validation OK - Shape : (7043, 27) | Churn rate : 26.5%


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,...,1,0,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,1,0,1,...,1,0,0,0,1,0,0,0,0,1
2,1,0,0,0,2,1,0,1,1,0,...,1,0,0,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,1,0,1,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,0,2,1,0,0,0,0,...,0,1,0,1,0,0,0,0,1,0


## 7. Sauvegarde

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/telco_clean.csv', index=False)
print('Données nettoyées et sauvegardées dans le fichier telco_clean.csv')

# Sauvegarder df_eda : avec labels originaux pour la visualisation
df_eda = df_raw.copy()
df_eda['Churn_num'] = (df_eda['Churn'] == 'Yes').astype(int)
df_eda.to_csv('../data/processed/telco_eda.csv', index=False)
print('Données EDA sauvegardées avec succée.')

Données nettoyées et sauvegardées dans le fichier telco_clean.csv
Données EDA sauvegardées avec succée.
